In [ ]:

import os
import io
import pandas as pd
import numpy as np 
import seaborn as sns
from tqdm import tqdm
import pickle

from pydub import AudioSegment, exceptions
AudioSegment.converter = "/usr/local/bin/ffmpeg"

from pydub.utils import make_chunks
import soundfile as sf
from IPython.display import Audio

import librosa
import librosa.display
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, accuracy_score
import itertools
import logging

import tensorflow as tf
from tensorflow.keras import layers, models

import warnings
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [ ]:
def convert_to_mono(audio):
    """Convert stereo audio to mono."""
    return audio.set_channels(1)

def get_split_unique(path, base_filename):
    """
    Splits an audio file into 1.5-second chunks and names each chunk uniquely.

    Args:
        path (str): The file path to the input audio file.
        base_filename (str): The base name to use for naming the chunk files.

    Returns:
        list: A list of in-memory audio chunks.
    """
    # Carga el archivo de audio
    audio = AudioSegment.from_file(path, "mp3")
    logging.debug(f"Audio: {path}")
    # Convierte a mono
    mono_audio = convert_to_mono(audio)
    logging.debug(f"Mono audio")
    
    # Define la longitud de cada trozo
    chunk_length_ms = 1500  # Tamaño de cada trozo (1.5 segundos en este ejemplo)

    # Divide el audio en trozos
    chunks = make_chunks(mono_audio, chunk_length_ms)

    # Almacena cada trozo en un objeto en memoria
    chunk_data = []
    for chunk in tqdm(chunks):
        chunk_io = io.BytesIO()
        chunk.export(chunk_io, format="mp3")
        chunk_data.append(chunk_io.getvalue())

    return chunk_data

def get_split_with_label(row):
    """
    Splits an audio file into chunks and associates each chunk with the audio's original label.

    Args:
        row (pd.Series): A row from a DataFrame containing 'audio_path' and 'label'.

    Returns:
        list of tuples: A list of tuples, where each tuple contains the in-memory chunk data and the label.
    """
    path = row['audio_path']
    label = row['label']
    base_filename = os.path.splitext(os.path.basename(path))[0]  # Use the base file name

    # Obtiene los trozos de audio
    chunk_data = get_split_unique(path, base_filename)

    # Empareja los trozos de audio con la etiqueta original
    return [(chunk, label) for chunk in chunk_data]

In [ ]:
def get_mfcc(path):
    y, sr = librosa.load(path, sr=None, mono=True)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmax=10000)
    mfccs = librosa.feature.mfcc(S=librosa.power_to_db(S), n_mfcc=40)
    return mfccs.T

def process_audio_files_batch(df, batch_size=64):
    logging.info(f"Iniciando procesamiento de audio")
    for start in range(0, len(df), batch_size):
        end = min(start + batch_size, len(df))
        batch_df = df[start:end]
        temp = []
        label = []

        for _, row in batch_df.iterrows():
            audio_path = row['audio_path']
            mfcc = get_mfcc(audio_path)
            mfcc = np.resize(mfcc, (65, 40))
            temp.append(mfcc)
            label.append(row['label'])

        X_processed = np.asarray(temp)
        y_processed = np.asarray(label)
        yield X_processed, y_processed

In [ ]:
# Add attention layer to the deep learning network
class Attention(layers.Layer):
    """
    Custom Keras Layer implementing an Attention mechanism.

    Methods
    -------
    build(input_shape)
        Initializes the weights and biases for the attention mechanism.
    
    call(x)
        Applies the attention mechanism to the input tensor `x` and returns the context vector.

    Parameters
    ----------
    input_shape : tuple
        Shape of the input tensor.
    
    x : tensor
        Input tensor to which the attention mechanism is applied.

    Returns
    -------
    tensor
        Context vector obtained after applying the attention mechanism.
    """
    def __init__(self, **kwargs):
        super(Attention, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name='attention_weight', shape=(input_shape[-1], 1), 
                                 initializer='random_normal', trainable=True)
        self.b = self.add_weight(name='attention_bias', shape=(input_shape[1], 1), 
                                 initializer='zeros', trainable=True)
        super(Attention, self).build(input_shape)

    def call(self, x):
        # Alignment scores. Pass them through tanh function
        e = tf.tanh(tf.matmul(x, self.W) + self.b)  # Cambiado K.dot por tf.matmul
        # Remove dimension of size 1
        e = tf.squeeze(e, axis=-1)  # Cambiado K.squeeze por tf.squeeze
        # Compute the weights
        alpha = tf.nn.softmax(e)  # Cambiado K.softmax por tf.nn.softmax
        # Reshape to tensorFlow format
        alpha = tf.expand_dims(alpha, axis=-1)  # Cambiado K.expand_dims por tf.expand_dims
        # Compute the context vector
        context = x * alpha
        context = tf.reduce_sum(context, axis=1)  # Cambiado K.sum por tf.reduce_sum
        return context

In [ ]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Model
import keras.backend as K

tf.keras.backend.clear_session()

input_csv = '/Users/camcortes/Documents/birds-sounds/data/paths_cantos_passeriformes.csv'
df = pd.read_csv(input_csv)
df = df.sample(128)

le = LabelEncoder()
df['label'] = le.fit_transform(df['Specie'])

chunk_data = df.apply(get_split_with_label, axis=1)
chunk_data = [item for sublist in chunk_data for item in sublist]
chunk_df = pd.DataFrame(chunk_data, columns=['audio_path', 'label'])

num_clases = len(chunk_df['label'].unique())

#Create model

#input
input = layers.Input(shape=(65,40), name='input_layer')
#cnn
x = layers.Conv1D(256, (3,), activation='relu')(input)
x = layers.BatchNormalization()(x)
x = layers.Conv1D(128, (3,), activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Conv1D(64, (3,), activation='relu')(x)
x = layers.BatchNormalization()(x)
#lstm
x = layers.LSTM(256, return_sequences = True)(x)
x = layers.Dropout(0.5)(x)
x = layers.LSTM(128, return_sequences = True)(x)
x = layers.Dropout(0.5)(x)
x = layers.LSTM(64, return_sequences = True)(x)
x = layers.Dropout(0.5)(x)
x = Attention()(x)
#dense
x = layers.Dense(256,'relu')(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(128,'relu')(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(64,'relu')(x)
x = layers.Dropout(0.5)(x)

#output
output = layers.Dense(num_clases, activation='softmax', name='output_layer')(x)

model = Model(inputs=input, outputs=output)
model.summary()

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

for X_batch, y_batch in process_audio_files_batch(chunk_df):
    model.fit(X_batch, y_batch, epochs=1, batch_size=16)